#Configuración del Entorno, Partición Estratificada y Generación de Datos Aumentados
###1. ¿Qué se hizo?
Se configuró el entorno de ejecución determinista (seed=42) en GPU, se cargaron los conjuntos train.csv y eval.csv, se aplicó un clasificador léxico ligero (detect_lang) para etiquetar el idioma dominante de cada reseña (es, en, pt, fr o mezcla es-en), y se construyó una partición estratificada conjunta por clase e idioma (83.3% entrenamiento / 16.7% validación mediante StratifiedKFold(n_splits=6)). Posteriormente, exclusivamente sobre el subconjunto de entrenamiento (trn_df), se generaron 4,450 reseñas sintéticas mediante tres estrategias complementarias:

* Code-Switching Sintético (aug_code_switching): 1,350 reseñas híbridas (450 por
clase) que intercalan oraciones en español e inglés de reseñas reales con la misma polaridad.

* Traducción Cruzada Neuronal (aug_translation): 1,500 reseñas polares (negativo y positivo) traducidas desde español hacia inglés (Helsinki-NLP/opus-mt-es-en) y francés (Helsinki-NLP/opus-mt-es-fr).

*Perturbación Léxica Protegida (aug_protected_dropout): ~1,600 variaciones de reseñas difíciles con conectores adversativos (pero, aunque, sin embargo, but, however, mas, mais), eliminando una palabra descriptiva larga sin tocar negaciones ni conectores de contraste.

###2. ¿Por qué se decidió hacer eso?
* Estratificación conjunta (label + lang): Dado que eval.csv tiene una proporción mayor de inglés, portugués, francés y code-switching que train.csv, un split aleatorio simple sobrerrepresentaría el español en validación y ocultaría fallas en idiomas minoritarios.

* Aumentación dirigida a partir del diagnóstico de errores: En la iteración previa (xlm-roberta-large sin aumentación), la clase neutral ya alcanzaba un F1 de 0.9979, mientras que los errores se concentraban en la frontera negativo vs. positivo (concesiones e ironía), en code-switching (es-en = 0.6818 de accuracy), en inglés (0.8824) y en francés (0.8931). Por ello, se focalizó la generación sintética en esos subgrupos exactos en lugar de aumentar todo el dataset de manera uniforme.

* Aislamiento estricto de validación: Aumentar únicamente trn_df después del split evita cualquier fuga de datos (data leakage), ya que una reseña traducida o perturbada nunca aparece en val_df.

###3. ¿Qué otras alternativas había y por qué no se hicieron?
* EDA clásico (Easy Data Augmentation: inserción/intercambio aleatorio de sinónimos sin restricciones): Se descartó porque en reseñas con ironía o negaciones de largo alcance ("no diría que la batería sea mala"), eliminar o alterar al azar un adverbio de negación o un conector adversativo invierte la polaridad real del texto e introduce ruido de etiquetado.

* Oversampling aleatorio simple (duplicación exacta de filas): No aporta diversidad léxica ni sintáctica; el Transformer memorizaría las mismas secuencias en lugar de aprender representaciones invariantes al cambio de idioma.

###4. ¿Qué cambia frente al código anterior (Iteración #2)?
En la Iteración #2 (xlm-roberta-large sin aumentación), el modelo se entrenó únicamente con los 17,055 ejemplos originales de trn_df.

En esta iteración, se mantienen exactamente las mismas particiones trn_df y val_df (seed=42) para garantizar una comparación controlada, pero el conjunto de entrenamiento crece de 17,055 a ~21,505 ejemplos (+26.1%) enriquecidos en es-en, en, fr y estructuras adversativas, además de incorporar una columna aug_type para auditar el origen de cada muestra.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn sentencepiece sacremoses

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    MarianMTModel, MarianTokenizer, get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Usando dispositivo:", device)

train_df = pd.read_csv('train.csv', encoding='utf-8')
eval_df = pd.read_csv('eval.csv', encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

def detect_lang(text):
    t = text.lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
eval_df['lang'] = eval_df['text'].apply(detect_lang)

# Mismo split exacto que el baseline sin aumentación
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(train_df, train_df['strat_key']):
    trn_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    break

# --- ESTRATEGIAS DE AUMENTACIÓN DE DATOS (SOLO EN trn_df) ---

# 1. Code-Switching Sintético (ES <-> EN manteniendo etiqueta)
def aug_code_switching(df, n_per_class=450):
    rows = []
    for label in ['negativo', 'positivo', 'neutral']:
        es_list = df[(df['lang'] == 'es') & (df['label'] == label)]['text'].tolist()
        en_list = df[(df['lang'] == 'en') & (df['label'] == label)]['text'].tolist()
        for _ in range(n_per_class):
            s_es = [s.strip() for s in re.split(r'(?<=[.!?])\s+', random.choice(es_list)) if len(s.strip()) > 6]
            s_en = [s.strip() for s in re.split(r'(?<=[.!?])\s+', random.choice(en_list)) if len(s.strip()) > 6]
            if len(s_es) >= 2 and len(s_en) >= 2:
                mixed = []
                for i in range(min(max(len(s_es), len(s_en)), 4)):
                    if i % 2 == 0 and i < len(s_es):
                        mixed.append(s_es[i])
                    elif i < len(s_en):
                        mixed.append(s_en[i])
                rows.append({'text': ' '.join(mixed), 'label': label, 'lang': 'es-en', 'aug_type': 'code_switching'})
    return pd.DataFrame(rows)

# 2. Traducción Cruzada Neuronal (ES -> EN y ES -> FR en reseñas polares)
def aug_translation(df, model_name, target_lang, n_samples=750):
    tok = MarianTokenizer.from_pretrained(model_name)
    mod = MarianMTModel.from_pretrained(model_name).to(device)
    mod.eval()
    sub = df[(df['lang'] == 'es') & (df['label'].isin(['negativo', 'positivo']))].sample(n=n_samples, random_state=42)
    texts = sub['text'].tolist()
    labels = sub['label'].tolist()
    out_texts = []
    with torch.no_grad():
        for i in range(0, len(texts), 32):
            enc = tok(texts[i:i+32], return_tensors='pt', padding=True, truncation=True, max_length=160).to(device)
            gen = mod.generate(**enc, max_length=160, num_beams=2)
            out_texts.extend(tok.batch_decode(gen, skip_special_tokens=True))
    del mod, tok
    torch.cuda.empty_cache()
    return pd.DataFrame({'text': out_texts, 'label': labels, 'lang': target_lang, 'aug_type': f'trans_{target_lang}'})

# 3. Perturbación Léxica Protegida (Word Dropout suave sin borrar negaciones ni conectores adversativos)
PROTECTED = {
    'no','nunca','tampoco','ni','sin','pero','aunque','embargo','sino','nada','malo','mala','bueno','buena',
    'not','never','no','without','but','although','however','yet','bad','good','great','poor',
    'não','nunca','sem','mas','porém','embora','ruim','bom','boa',
    'pas','jamais','sans','mais','cependant','bien','bon','mauvais'
}

def aug_protected_dropout(df, n_samples=1600):
    mask = df['text'].str.lower().str.contains(r'\b(pero|aunque|sin embargo|but|however|mas|porém|mais)\b', regex=True)
    cand = df[mask].sample(n=min(n_samples, mask.sum()), random_state=42)
    rows = []
    for _, r in cand.iterrows():
        words = r['text'].split()
        elig = [i for i, w in enumerate(words) if re.sub(r'\W+', '', w.lower()) not in PROTECTED and len(w) > 4]
        if len(elig) >= 2:
            w_copy = words.copy()
            w_copy.pop(random.choice(elig))
            rows.append({'text': ' '.join(w_copy), 'label': r['label'], 'lang': r['lang'], 'aug_type': 'protected_dropout'})
    return pd.DataFrame(rows)

df_cs = aug_code_switching(trn_df, n_per_class=450)
df_en = aug_translation(trn_df, 'Helsinki-NLP/opus-mt-es-en', 'en', n_samples=750)
df_fr = aug_translation(trn_df, 'Helsinki-NLP/opus-mt-es-fr', 'fr', n_samples=750)
df_dr = aug_protected_dropout(trn_df, n_samples=1600)

trn_df['aug_type'] = 'original'
trn_aug = pd.concat([trn_df[['text', 'label', 'lang', 'aug_type']], df_cs, df_en, df_fr, df_dr], ignore_index=True)
trn_aug['target'] = trn_aug['label'].map(label2id)
trn_aug = trn_aug.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Distribución de datos aumentados:")
print(trn_aug['aug_type'].value_counts())
print(f"Total entrenamiento: {len(trn_df)} -> {len(trn_aug)} | Validación intacta: {len(val_df)}")

Usando dispositivo: cuda


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (16 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/tmp/ipykernel_6046/1441787282.py:108: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df['text'].str.lower().str.contains(r'\b(pero|aunque|sin embargo|but|however|mas|porém|mais)\b', regex=True)


Distribución de datos aumentados:
aug_type
original             17055
protected_dropout     1600
code_switching        1338
trans_fr               750
trans_en               750
Name: count, dtype: int64
Total entrenamiento: 17055 -> 21493 | Validación intacta: 3411


#Tokenización, Arquitectura (xlm-roberta-large) y Entrenamiento Regularizado
###1. ¿Qué se hizo?
Se tokenizaron las reseñas con AutoTokenizer de xlm-roberta-large usando longitud máxima MAX_LEN = 192 tokens, y se entrenó el modelo durante 4 épocas con batch size 16, tasa de aprendizaje 1e-5 (ajustada mediante cosine learning rate schedule con 10% de warmup), recorte de gradientes (clip_grad_norm_ = 1.0), precisión mixta (fp16 con GradScaler) y función de pérdida CrossEntropyLoss(label_smoothing=0.03). En cada época se evaluó el conjunto de validación intacto y, cuando el accuracy mejoró, se guardaron tanto los pesos del modelo (best_xlmr_large_aug.pt) como la matriz de probabilidades de validación (val_probs_xlmr_large_aug.npy).
###2. ¿Por qué se decidió hacer eso?
* xlm-roberta-large (560M parámetros, 24 capas): Supera significativamente a la versión base en representaciones multilingües cruzadas, lo que resulta crítico cuando una misma reseña cambia de idioma a mitad de párrafo (code-switching) o utiliza expresiones idiomáticas indirectas.
* MAX_LEN = 192: El análisis exploratorio mostró que el 99% de las reseñas tienen $\le 92$ palabras (máximo 125). Fijar 192 sub-tokens asegura que ninguna reseña sea truncada (evitando perder el veredicto final cuando hay concesiones al inicio) sin desperdiciar memoria VRAM en padding excesivo.
* Label Smoothing = 0.03: Al incorporar datos sintéticos traducidos y reseñas genuinamente ambiguas, suavizar ligeramente las etiquetas objetivo ([0.01, 0.98, 0.01] en lugar de [0, 1, 0]) previene el sobreajuste (overfitting) y mejora la calibración de las probabilidades predichas.
* Guardado de probabilidades (val_probs_xlmr_large_aug.npy): Almacenar la distribución softmax completa (y no solo la clase argmax) permite combinar posteriormente este modelo con otras arquitecturas mediante soft voting ponderado.
###3. ¿Qué otras alternativas había y por qué no se hicieron?
* Entrenar con BATCH_SIZE = 32 y LR = 2e-5 (como en xlm-roberta-base): Un modelo large con secuencias de 192 tokens desbordaría los 15–16 GB de VRAM de una GPU T4 gratuita en Colab con batch 32, y tasas de aprendizaje altas ($\ge 2\text{e-}5$) suelen desestabilizar las capas profundas de RoBERTa-large en las primeras iteraciones.
* Redes recurrentes (BiLSTM / GRU) o CNN desde cero: Aunque son más ligeras, carecen de conocimiento semántico preentrenado en 100 idiomas; para capturar ironía y code-switching con apenas ~20 mil muestras, la transferencia de aprendizaje con Transformers multilingües ofrece una capacidad de generalización muy superior.
* Congelar el backbone y entrenar solo la cabeza de clasificación: Limitaría la adaptación del espacio latente multilingüe a las sutilezas de opinión indirecta del dominio de comercio electrónico.
###4. ¿Qué cambia frente al código anterior (Iteración #2)?
* Conjunto de entrenamiento: El DataLoader recibe trn_aug (~21,505 filas) en lugar de trn_df (17,055 filas).
* Función de pérdida: Se reemplaza la pérdida interna por defecto del modelo (entropía cruzada estándar) por un criterio explícito nn.CrossEntropyLoss(label_smoothing=0.03) para regularizar el aprendizaje sobre muestras aumentadas.
* Salida de validación: En lugar de descartar las probabilidades y guardar solo argmax, se aplica torch.softmax(outputs.logits, dim=1) y se exporta val_probs_xlmr_large_aug.npy junto con el checkpoint best_xlmr_large_aug.pt.

In [ ]:
MODEL_NAME = 'xlm-roberta-large'
MAX_LEN = 192
BATCH_SIZE = 16
EPOCHS = 4
LR = 1e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

trn_ds = ReviewDataset(trn_aug['text'], trn_aug['target'].values)
val_ds = ReviewDataset(val_df['text'], val_df['target'].values)
eval_ds = ReviewDataset(eval_df['text'])

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(trn_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in trn_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            val_probs.append(probs)
            val_targets.extend(batch['labels'].numpy())

    val_probs = np.vstack(val_probs)
    val_preds = np.argmax(val_probs, axis=1)
    val_acc = accuracy_score(val_targets, val_preds)
    print(f"Época {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(trn_loader):.4f} | Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_xlmr_large_aug.pt')
        np.save('val_probs_xlmr_large_aug.npy', val_probs)
        val_df['pred_aug'] = val_preds

print(f"\nMejor Val Accuracy (Con Aumentación): {best_val_acc:.4f} (vs Sin Aumentación: 0.9074)")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Época 1/4 | Train Loss: 0.5110 | Val Accuracy: 0.8959
Época 2/4 | Train Loss: 0.2976 | Val Accuracy: 0.9035
Época 3/4 | Train Loss: 0.2508 | Val Accuracy: 0.9118
Época 4/4 | Train Loss: 0.2138 | Val Accuracy: 0.9074

Mejor Val Accuracy (Con Aumentación): 0.9118 (vs Sin Aumentación: 0.9074)


In [ ]:
print("=== Reporte Global en Validación (xlm-roberta-large + Aumentación) ===")
print(classification_report(val_df['target'], val_df['pred_aug'], target_names=['negativo', 'neutral', 'positivo'], digits=4))

# Comparación directa contra los resultados sin aumentación de tu compañero
base_lang_acc = {'en': 0.8824, 'es': 0.9145, 'es-en': 0.6818, 'fr': 0.8931, 'other': 1.0000, 'pt': 0.9160}
print("=== Comparación por Idioma (Sin Aug vs Con Aug) ===")
for lang, grp in val_df.groupby('lang'):
    acc_aug = accuracy_score(grp['target'], grp['pred_aug'])
    acc_base = base_lang_acc.get(lang, 0.0)
    print(f"Idioma {lang:6s} (n={len(grp):4d}): Sin Aug = {acc_base:.4f} | Con Aug = {acc_aug:.4f} | Delta = {acc_aug - acc_base:+.4f}")

model.load_state_dict(torch.load('best_xlmr_large_aug.pt'))
model.eval()
eval_probs = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        eval_probs.append(torch.softmax(outputs.logits, dim=1).cpu().numpy())

eval_probs = np.vstack(eval_probs)
np.save('eval_probs_xlmr_large_aug.npy', eval_probs)

eval_preds = np.argmax(eval_probs, axis=1)
submission_3 = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_preds]
})

submission_3.to_csv('submission_3_xlmr_large_aug.csv', index=False)
print("\nArchivo generado: submission_3_xlmr_large_aug.csv (y probabilidades guardadas para ensamble)")
print(submission_3['answer'].value_counts(normalize=True).round(4))

=== Reporte Global en Validación (xlm-roberta-large + Aumentación) ===
              precision    recall  f1-score   support

    negativo     0.9182    0.8301    0.8719      1230
     neutral     1.0000    0.9968    0.9984       932
    positivo     0.8467    0.9287    0.8858      1249

    accuracy                         0.9118      3411
   macro avg     0.9216    0.9185    0.9187      3411
weighted avg     0.9144    0.9118    0.9116      3411

=== Comparación por Idioma (Sin Aug vs Con Aug) ===
Idioma en     (n= 391): Sin Aug = 0.8824 | Con Aug = 0.8824 | Delta = -0.0000
Idioma es     (n=2282): Sin Aug = 0.9145 | Con Aug = 0.9189 | Delta = +0.0044
Idioma es-en  (n=  22): Sin Aug = 0.6818 | Con Aug = 0.6364 | Delta = -0.0454
Idioma fr     (n= 346): Sin Aug = 0.8931 | Con Aug = 0.9075 | Delta = +0.0144
Idioma other  (n=   1): Sin Aug = 1.0000 | Con Aug = 1.0000 | Delta = +0.0000
Idioma pt     (n= 369): Sin Aug = 0.9160 | Con Aug = 0.9187 | Delta = +0.0027

Archivo generado: submissio

El Score conseguido en Kaggle fue de 0.89047, lo que es menor frente a la iteración #2 (roberta-large Sin aumentación) que gue de 0.89685, creemos que lo que fallo fue:
###1. Desplazamiento del sesgo de predicción (Class Prior Shift):

En la Iteración #2 (0.89685), xlm-roberta-large predijo en eval.csv: positivo: 42.37% vs. negativo: 38.58%.

En la Iteración #3 (0.89047), al combinar label smoothing (0.03) con traducción cruzada y mezcla de oraciones, el modelo desplazó su frontera hacia positivo (45.80% vs. 35.23% en negativo), cayendo el recall de negativo de 0.8699 a 0.8301. En eval.csv (que tiene más ejemplos difíciles con concesiones e ironía), volverse más optimista penalizó ~43 predicciones.

###2. Ruido semántico por mezcla de oraciones (code_switching entre reseñas distintas) y traducción automática:

Pegar oraciones de dos reseñas diferentes (ej. hablar de audífonos en español y luego de una licuadora en inglés) crea textos artificiales fuera de distribución.

Las traducciones automáticas (MarianMT) suavizan modismos e ironías sutiles, haciendo que el modelo confíe más en palabras positivas aisladas del inicio de la reseña.